# Exercise 2 · Approve one fix, then check it with /diff and the tests

You will approve one fix from Codex after you read it, and then check it yourself.

**Goal:** Make Codex ask before it edits, approve one change after you read it, and confirm the change with `/diff`, the tests and Git.

**Why it matters:** Codex can change many lines in seconds; reading each change first keeps you in charge of your code.

**From the docs:** [Agent approvals & security](https://learn.chatgpt.com/codex/agent-approvals-security), [Sandbox](https://learn.chatgpt.com/codex/sandboxing), [Auto-review](https://learn.chatgpt.com/codex/sandboxing/auto-review), [Code review](https://learn.chatgpt.com/codex/code-review?surface=cli), [Developer commands](https://learn.chatgpt.com/codex/developer-commands?surface=cli), [Best practices](https://learn.chatgpt.com/codex/learn/best-practices), [Codex CLI](https://learn.chatgpt.com/docs/codex/cli).

**Before you start:**
- Codex, a coding agent that reads, runs and edits code from your terminal, is installed and signed in.
- A test is a small function that checks code; `python -m pytest -q` runs the tests in a folder.
- A Git commit saves every file as it is, so you can later see what changed and go back.

**What the check measures:** the tests are unchanged; `apply_discount(50.0, 10)` is `45.0` and `apply_discount(80.0, 25)` is `60.0`; the other functions still work; and the tests pass.

**Time:** about 25 minutes.

**Done when:** the last cell prints `Done:`.

## The situation

A small shopping-list app has a bug: 10 percent off 50.00 gives 40.00, not 45.00. You ask Codex to fix it, but it may change nothing until you read the change and say yes. The files are made up; never give Codex real or sensitive data.

## Step 1 · Write the practice files

Run the cell. It writes three files into `practice/`, beside this notebook: `shopping.py`, where `apply_discount` has the bug; `test_shopping.py`, one test per function; and `.gitignore`, which keeps Python's cache folders out of Git.

Expect three lines that start with `Wrote practice/`, or `Kept` for a file that is already there.

In [ ]:
# Step 1 (solved copy): write practice/ as it is after step 11
from pathlib import Path

PRACTICE = Path("practice")
FILES = {
    "shopping.py": '''"""Totals for a shopping list."""


def line_total(price: float, quantity: int) -> float:
    """Return the cost of one line: price times quantity."""
    return price * quantity


def basket_total(lines: list[tuple[float, int]]) -> float:
    """Return the cost of all lines, rounded to cents."""
    return round(sum(line_total(price, quantity) for price, quantity in lines), 2)


def apply_discount(total: float, percent: float) -> float:
    """Return total with percent taken off, rounded to cents: 10 takes off 10%."""
    return round(total - percent, 2)
''',
    "test_shopping.py": '''from shopping import apply_discount, basket_total, line_total


def test_line_total():
    assert line_total(2.5, 4) == 10.0


def test_basket_total():
    assert basket_total([(2.5, 4), (1.25, 2)]) == 12.5


def test_apply_discount():
    assert apply_discount(50.0, 10) == 45.0
''',
    ".gitignore": "__pycache__/\n.pytest_cache/\n",
}
SOLVED = {**FILES, "shopping.py": FILES["shopping.py"].replace(
    "round(total - percent, 2)", "round(total * (1 - percent / 100), 2)")}

PRACTICE.mkdir(exist_ok=True)
for name, text in SOLVED.items():
    if (PRACTICE / name).exists():
        print(f"Kept practice/{name}")
    else:
        (PRACTICE / name).write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

## Step 2 · Open a terminal in the practice folder

Choose **Terminal > New Terminal**; it opens in the course folder. Type these two lines, pressing Enter after each; the first selects the course's Python.

```sh
source .venv/bin/activate
cd week-01/day-special/exercise-02-approve-one-fix/practice
```

Expect the prompt, the text before your cursor, to start with `(practice-exercises)` and to show `practice`.

### Learn: Git lets you undo an agent's edits

The Codex CLI has no undo command; Git is your safety net. A commit saves every file as it is; `/diff` then shows what changed, and `git restore` puts a file back. Commit before and after each task, "so you can revert changes" ([Codex CLI](https://learn.chatgpt.com/docs/codex/cli), "Start your first task").

## Step 3 · Commit the starting files

This makes `practice/` a Git repository, a folder whose history Git keeps, and commits the three files as `Start`:

```sh
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect no output: `-q` keeps Git quiet.

## Step 4 · Run the tests yourself

See the bug yourself first:

```sh
python -m pytest -q
```

Expect `1 failed, 2 passed`. Above it, `assert 40.0 == 45.0` shows the bug: `apply_discount(50.0, 10)` takes off 10 as money, not 10 percent.

### Learn: the sandbox and the approval policy

Two settings decide what Codex may do ([Agent approvals & security](https://learn.chatgpt.com/codex/agent-approvals-security), "Common sandbox and approval combinations"):

- The **sandbox** limits what Codex can touch. In `read-only`, "Codex can read files and run commands within the read-only sandbox. Actions outside the sandbox can require approval." Editing a file is outside it.
- The **approval policy** sets when Codex asks you: with `on-request`, before it steps outside the sandbox.

In a Git folder, Codex suggests Auto (`workspace-write` with `on-request`), which edits files without asking ("Defaults and recommendations").

## Step 5 · Start Codex so it must ask before it edits

```sh
codex --no-daemon --sandbox read-only --ask-for-approval on-request
```

`--no-daemon` runs Codex's commands in this terminal, with the course's Python. If your Codex does not know `--no-daemon`, leave it out.

If Codex asks "Trust this folder?", press Enter on `1. Trust and continue`: the folder holds only the practice files.

Expect `>_ OpenAI Codex` at the top, with a folder that ends in `practice`.

## Step 6 · Check the mode with /status

```text
/status
```

Expect a `Permissions:` line such as `Read Only (Ask for approval)`: read-only, with on-request approval. If it says otherwise, type `/exit` and start Codex again as in step 5.

### Learn: one small request, with a done-when

A good request has four parts ([Best practices](https://learn.chatgpt.com/codex/learn/best-practices), "Strong first use: Context and prompts"): the **goal**; the **context**, such as the file and the error; the **constraints**; and **done when**: "What should be true before the task is complete, such as tests passing".

## Step 7 · Ask for one small fix

Type the request below; where it says `@shopping.py`, type `@shopping` and pick `shopping.py` from the list (`@` finds a file and adds its path). Press Enter.

```text
Fix apply_discount in @shopping.py: test_apply_discount fails because apply_discount(50.0, 10) returns 40.0, not 45.0. Change only shopping.py, keep the fix small, and don't edit the tests. Done when python -m pytest -q passes.
```

Expect Codex to read the files, then stop and ask before it edits `shopping.py`.

### Learn: read first, then approve once

In read-only, Codex must stop and show you each edit, or the command that makes it, before it runs. The choices depend on your Codex version; the docs say: "choose the narrowest scope that lets the task continue" ([Sandbox](https://learn.chatgpt.com/codex/sandboxing), "How permissions work"). Once is narrower than for the whole session.

With **Approve for me**, a reviewer agent, a second Codex, answers for you ([Auto-review](https://learn.chatgpt.com/codex/sandboxing/auto-review)).

## Step 8 · Read the change, then approve it once

In Codex's request, only the `return` line of `apply_discount` should change, to take `percent` off as a share of `total`, such as `total * (1 - percent / 100)`. If so, approve it once; if not, decline and say what to do instead. If Codex asks to read files or run the tests, approve that once too. Codex may ask more than once: read each request before you approve it.

Expect Codex to say the tests pass. If it never asked, your account uses Approve for me, and step 9 is your only review.

<details><summary>Hint</summary>

10 percent of 50.0 is 5.0, so the result must be 45.0. An option that says "don't ask again" covers more than this request.

</details>

<details><summary>Show the answer</summary>

Approve once if the new line is `return round(total * (1 - percent / 100), 2)`, or the same sum written another way; decline anything else. Why: a one-time yes lets only this change through.

</details>

### Learn: review the change before you keep it

`/diff` shows what changed since your last commit, new files included: `-` marks a line taken out, `+` a line put in. Use it to "Review Codex's edits before you commit or run tests." ([Developer commands](https://learn.chatgpt.com/codex/developer-commands?surface=cli), "Built-in slash commands"). `/review` adds a second opinion ([Code review](https://learn.chatgpt.com/codex/code-review?surface=cli)).

## Step 9 · Review the change with /diff

```text
/diff
```

After a few seconds, the diff fills the screen. Read it, then press `q` to close it. Expect only `shopping.py`: one line out, one line in. Your `+` line may write the same sum another way:

```text
-    return round(total - percent, 2)
+    return round(total * (1 - percent / 100), 2)
```

## Step 10 · Leave Codex and run the tests

Close the diff with `q` if it is still open. Leave Codex with `/exit`, then run the tests:

```text
/exit
python -m pytest -q
```

Expect `3 passed`. If not, or if the change looks wrong, `git restore shopping.py` brings back your `Start` file; go back to step 5.

## Step 11 · Commit the fix

Commit the change, then list your commits:

```sh
git commit -qam "Fix apply_discount"
git log --oneline
```

Expect two lines, newest first: `Fix apply_discount`, then `Start`, each after a short ID.

## Start over (only if you need it)

To start over, set `START_OVER = True` and run the cell: it rewrites the practice files as Step 1 wrote them, and keeps your Git history. Then set it back to `False` and go to step 3, where Git may print `re-init` or `nothing to commit`: both are fine.

Expect `Start over is off: no file changed.` while `START_OVER` is `False`.

In [ ]:
# Start over: rewrite the practice files as Step 1 wrote them
START_OVER = False  # Set to True only to start over.

if not START_OVER:
    print("Start over is off: no file changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first.")
else:
    for name, text in FILES.items():
        (PRACTICE / name).write_text(text, encoding="utf-8")
        print(f"Rewrote practice/{name}")
    print("Now set START_OVER back to False.")

## Check your work

Run the cell.

Expect `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do.

In [ ]:
# Check your work: read practice/, try the functions, and run the tests
import contextlib
import io
import runpy
import sys

import pytest

UNDO = "Undo it with the Start over cell, then go to step 3."
CASES = [  # (function, arguments, right result, problem if not)
    ("apply_discount", (50.0, 10), 45.0, "the new line is wrong"),
    ("apply_discount", (80.0, 25), 60.0, "the fix fits only the one test"),
    ("line_total", (2.5, 4), 10.0, "line_total or basket_total changed"),
    ("basket_total", ([(2.5, 4), (1.25, 2)],), 12.5, "line_total or basket_total changed"),
]


def first_problem() -> str:
    """Return the first problem, or "" if none; print ok for each check that passes."""
    if (PRACTICE / "test_shopping.py").read_text(encoding="utf-8") != FILES["test_shopping.py"]:
        return ("the tests changed, but step 7 says not to edit them. "
                "Delete practice/test_shopping.py, then run the Step 1 cell and this cell.")
    print("ok: the tests are as Step 1 wrote them")
    try:
        shop = runpy.run_path(str(PRACTICE / "shopping.py"))
        results = [shop[name](*arguments) for name, arguments, _, _ in CASES]
    except Exception as error:
        return f"practice/shopping.py fails ({type(error).__name__}: {error}). {UNDO}"
    for (name, arguments, right, wrong), got in zip(CASES, results):
        call = f"{name}({', '.join(map(repr, arguments))})"
        if call == "apply_discount(50.0, 10)" and got == 40.0:
            return f"{call} gives 40.0, not 45.0: shopping.py is not fixed yet. Do steps 2 to 11."
        if got != right:
            return f"{call} gives {got}, not {right}: {wrong}. {UNDO}"
        print(f"ok: {call} is {right}")
    for name in ("shopping", "test_shopping"):  # so that pytest reads the files again
        sys.modules.pop(name, None)
    with contextlib.redirect_stdout(io.StringIO()) as out, contextlib.redirect_stderr(io.StringIO()):
        code = pytest.main(["-q", "--color=no", "-p", "no:cacheprovider", str(PRACTICE)])
    if code != 0:
        last = out.getvalue().strip().splitlines()[-1]
        return f"the tests fail ({last}). Run python -m pytest -q in practice/ to see why."
    print("ok: the tests pass")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run the Step 1 cell, then this cell."
print(f"Not yet: {problem}" if problem else
      "Done: apply_discount takes percent off as a share, and the unchanged tests pass.")

## What you learned

- Read-only with on-request approval makes Codex ask before it edits; give the narrowest yes ([Sandbox](https://learn.chatgpt.com/codex/sandboxing)).
- A goal, context, constraints and a done-when keep a request small and checkable ([Best practices](https://learn.chatgpt.com/codex/learn/best-practices)).
- Commits before and after a task, plus `/diff` and a test run, show what changed and let you undo it ([Codex CLI](https://learn.chatgpt.com/docs/codex/cli)).

**Reflect:**

- With Approve for me, which changes would you still want to check yourself?
- A fix can pass the one test and still be wrong. What test would you add?